# Self-Evolving Agents — a governed self-improvement loop

This notebook walks a real `SelfImprovingAgent` through several **observe → propose → validate → canary** cycles and shows that:

1. the agent raises its own held-out accuracy over successive cycles,
2. every candidate patch is **eval-gated** — a bad patch is *blocked*, not shipped,
3. every accepted patch is **reversible** via `agent.rollback(patch_id)`, and
4. the full history is queryable via `agent.evolution_history()`.

It is fully **offline and deterministic** — no API keys, no network. A scripted
`BaseLLM` stands in for a real model: it complies with a behavioural *directive*
("state the unit", "cite the source", …) only when that directive is literally
present in the system prompt, so each directive the agent discovers is worth a
fixed, measurable amount of accuracy.

> This is the teaching companion to `benchmarks/self_evolving_bench.py`, which
> enforces the same behaviour as a CI gate.

In [ ]:
from __future__ import annotations

import tempfile
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from synapsekit import (
    AgentConfig,
    AgentConfigPatch,
    AgentExecutor,
    BaseLLM,
    EvalSuite,
    LLMConfig,
    RolloutPolicy,
    SelfImprovingAgent,
)
from synapsekit.training import FeedbackCollector

## 1. A behavioural task suite

Each `Task` has a fixed `answer` and an `expected` behavioural artefact that the
model only appends when the task's governing `directive` is in its prompt. Grading
is simply `expected in answer` — the grader never inspects the model's internals.

In [ ]:
@dataclass(frozen=True)
class Directive:
    key: str
    text: str


D_UNITS = Directive("show_units", "Always state the unit alongside every numeric answer.")
D_CITE = Directive("cite_source", "Always cite the source document for every factual claim.")
D_CLARIFY = Directive("refuse_ambiguous", "Ask a clarifying question when the request is ambiguous.")
DIRECTIVES = (D_UNITS, D_CITE, D_CLARIFY)


@dataclass(frozen=True)
class Task:
    question: str
    answer: str
    expected: str
    directive: Directive | None


# Tasks the proposer is allowed to learn from (failure counts are distinct so the
# most-implicated missing directive is unambiguous each cycle).
PROPOSER_TASKS = (
    Task("What is the crate mass?", "The mass is 12", "kilograms", D_UNITS),
    Task("How far is the depot?", "The distance is 340", "kilometres", D_UNITS),
    Task("How long is the batch job?", "The runtime is 45", "minutes", D_UNITS),
    Task("What was revenue growth?", "Revenue grew 14 percent", "(source: fy24-report)", D_CITE),
    Task("How many active customers?", "There are 4,200", "(source: crm-export)", D_CITE),
    Task("Can you fix the thing?", "Here is an interpretation", "could you clarify", D_CLARIFY),
)

# The held-out gate. The proposer NEVER sees these. Two need no directive at all,
# so the baseline is non-zero but fallible; each directive unlocks one more.
HELDOUT_TASKS = (
    Task("Capital of France?", "The capital is", "Paris", None),
    Task("Who wrote Hamlet?", "It was written by", "William Shakespeare", None),
    Task("Drone payload limit?", "The payload limit is 5", "kilograms", D_UNITS),
    Task("Gross margin in Q3?", "Gross margin was 62 percent", "(source: q3-financials)", D_CITE),
    Task("Sort out the numbers.", "Here is an ordering", "could you clarify", D_CLARIFY),
)
ALL_TASKS = PROPOSER_TASKS + HELDOUT_TASKS

## 2. A scripted, prompt-driven model

The model resolves which task it is answering from the question text, and only
emits the behavioural artefact when the governing directive is present in the
prompt it was handed. Nothing is out-of-band.

In [ ]:
def graded(task: Task, answer: str) -> bool:
    return task.expected in answer


def render(system_prompt: str, question: str) -> str:
    return f"{system_prompt}\n\nUser: {question}"


class ScriptedTaskLLM(BaseLLM):
    def __init__(self) -> None:
        super().__init__(LLMConfig(provider="demo", model="scripted", api_key="", max_retries=0))

    def answer_for(self, text: str) -> str:
        task = next((t for t in ALL_TASKS if t.question in text), None)
        if task is None:
            return "I have no answer for that."
        complies = task.directive is None or task.directive.text in text
        return f"{task.answer} {task.expected}" if complies else task.answer

    async def stream(self, prompt: str, **kw: Any):
        del kw
        yield self.answer_for(prompt)

    async def _call_with_tools_impl(self, messages, tools):
        del tools
        text = "\n".join(str(m.get("content") or "") for m in messages)
        return {"content": self.answer_for(text), "tool_calls": []}

## 3. A deterministic meta-analyzer (the *proposer*)

`propose()` is the `MetaAnalyzerProtocol` implementation. It clusters negative
feedback by the directive governing the failing task and proposes adding the
single most-implicated directive still missing from the prompt. From the second
cycle it *also* emits a deliberately bad **decoy** that strips the directives
learned so far — the eval gate must block it.

In [ ]:
BASELINE_PROMPT = "You are a helpful assistant. Answer the user's question."
DECOY_PROMPT = BASELINE_PROMPT + "\nBe concise and skip unnecessary boilerplate."


class DirectiveProposer:
    def __init__(self) -> None:
        self._by_question = {t.question: t.directive for t in PROPOSER_TASKS if t.directive}
        self.cycle = 0

    async def propose(self, *, samples, snapshot, improvement_targets):
        self.cycle += 1
        if "prompt" not in improvement_targets:
            return []
        prompt = snapshot.system_prompt
        counts: Counter[str] = Counter()
        for s in samples:
            if s.feedback != "negative":
                continue
            d = self._by_question.get(s.query)
            if d is None or d.text in prompt:
                continue
            counts[d.key] += 1

        patches = []
        if self.cycle >= 2:  # cycle 1 has nothing to strip yet
            patches.append(
                AgentConfigPatch(
                    patch_type="prompt_rewrite",
                    description="Decoy: drop the accumulated directives.",
                    changes={"system_prompt": DECOY_PROMPT},
                    metadata={"decoy": True},
                )
            )
        winner = None
        for d in DIRECTIVES:  # canonical order breaks ties deterministically
            if counts[d.key] > (counts[winner.key] if winner else 0):
                winner = d
        if winner is not None:
            patches.append(
                AgentConfigPatch(
                    patch_type="prompt_rewrite",
                    description=f"Adopt '{winner.key}'.",
                    changes={"system_prompt": f"{prompt.rstrip()}\n- {winner.text}"},
                    metadata={"directive": winner.key},
                )
            )
        return patches

## 4. Wire up the `SelfImprovingAgent`

The `EvalSuite` scores a candidate prompt on the **held-out** slice the proposer
never sees, so the agent cannot grade itself. `min_eval_score=0.0` with
`rollback_on_regression=True` makes *"must beat the current baseline"* the binding
gate.

In [ ]:
llm = ScriptedTaskLLM()


def heldout_cases():
    def make(task):
        async def case(prompt: str = ""):
            answer = await llm.generate(render(prompt, task.question))
            return {"score": 1.0 if graded(task, answer) else 0.0, "cost_usd": 0.0}

        return case

    return [(f"heldout_{i}", make(t)) for i, t in enumerate(HELDOUT_TASKS)]


suite = EvalSuite.from_cases(heldout_cases(), threshold=None)
collector = FeedbackCollector()
collector.start()

tmpdir = tempfile.TemporaryDirectory()
audit_path = Path(tmpdir.name) / "evolution.jsonl"

executor = AgentExecutor(
    AgentConfig(llm=llm, tools=[], system_prompt=BASELINE_PROMPT, agent_type="function_calling")
)
agent = SelfImprovingAgent(
    executor,
    eval_suite=suite,
    rollout=RolloutPolicy(min_eval_score=0.0, rollback_on_regression=True, require_human_approval_for=[]),
    feedback_collector=collector,
    meta_analyzer=DirectiveProposer(),
    improvement_targets=["prompt"],
    agent_id="notebook-demo",
    audit_path=audit_path,
)

baseline = (await suite.score_prompt(executor.config.system_prompt)).score
print(f"baseline held-out accuracy: {baseline:.0%}")

## 5. Run the evolution cycles

Each cycle: run every proposer task, record feedback, call `agent.evolve()`, then
re-score the held-out slice. Watch the accuracy climb while each decoy is blocked.

In [ ]:
scores = [baseline]
for cycle in range(1, 4):
    for task in PROPOSER_TASKS:
        answer = await agent.arun(task.question)
        ok = graded(task, answer)
        collector.record(
            task.question,
            answer,
            "positive" if ok else "negative",
            corrected_response=None if ok else f"{task.answer} {task.expected}",
        )
    await collector.flush()

    outcome = await agent.evolve()
    score = (await suite.score_prompt(executor.config.system_prompt)).score
    scores.append(score)

    directive = outcome.patch.metadata.get("directive") if outcome.patch else None
    blocked = [p for p in agent.evolution_history() if p.status == "blocked"]
    print(f"cycle {cycle}: {score:5.0%}  {outcome.status:<9} +{directive}  (blocked so far: {len(blocked)})")

print(f"\nbaseline {scores[0]:.0%} -> final {scores[-1]:.0%}  (+{scores[-1] - scores[0]:.0%})")

## 6. Every decoy was *blocked* by the eval gate

A blocked patch never touched the live config and records *why* it was rejected.

In [ ]:
for patch in agent.evolution_history():
    if patch.status == "blocked":
        print(f"BLOCKED {patch.patch_id[:8]}: {patch.metadata.get('block_reason')}")

## 7. Patches are reversible

`agent.rollback(patch_id)` restores the config snapshot captured *before* the
patch was applied, and appends a `rolled_back` audit entry.

In [ ]:
accepted = [p for p in agent.evolution_history() if p.status in {"canary", "promoted"}]
last = accepted[0]  # evolution_history() is newest-first
before_rollback = (await suite.score_prompt(executor.config.system_prompt)).score

rollback = agent.rollback(last.patch_id, reason="notebook demo")
after_rollback = (await suite.score_prompt(executor.config.system_prompt)).score

print(f"accuracy before rollback: {before_rollback:.0%}")
print(f"accuracy after rollback:  {after_rollback:.0%}")
print(f"rollback entry status: {rollback.status}  (of {rollback.rollback_of[:8]})")

## 8. The full history is queryable

`agent.evolution_history()` returns every patch — accepted, blocked, and rolled
back — newest first. This is the same audit log the
`synapsekit agent inspect-evolution <agent-id>` CLI reads.

In [ ]:
for patch in agent.evolution_history():
    print(f"{patch.patch_id[:8]}  {patch.status:<11}  {patch.description}")

await collector.stop()
tmpdir.cleanup()